In [1]:
import spacy
import httpx
import time
import re
import os
import pandas as pd
from rapidfuzz import fuzz
from dotenv import load_dotenv

In [2]:
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_READ_ACCESS_TOKEN")
 
# check TMDb key is loaded
if not TMDB_API_KEY:
    print("WARNING: TMDB_READ_ACCESS_TOKEN not found in .env")
else:
    print("TMDb API key loaded")

TMDb API key loaded


In [ ]:
# load SpaCy model (using the transformer model, for more info see https://spacy.io/models/en)
nlp = spacy.load("en_core_web_trf")
print(f"SpaCy model loaded: {nlp.meta['name']}")

SpaCy model loaded: core_web_trf


In [ ]:
# test SpaCy NER on test posts
test_posts = [
    "I watched Severence yesterday and it was mid",
    "Cant wait to watch spider man tomorrow",
    "We finally watched Barbie lastg night and it was great!!!",
    "Anyone have movie recommendations for tonight?",
    "That film was incredible, no notes, 10/10",
    "Like my photo!? Shot on 35mm film",
    "Make your favourite film sound as boring as possible",
    "Finally watched sopranos",
    "watching my girlfriend eat soup out of a mug, wtf?"
]
 
print("SpaCy WORK_OF_ART extraction:\n")
for post in test_posts:
    doc = nlp(post)
    works = [ent.text for ent in doc.ents if ent.label_ == "WORK_OF_ART"]
    all_ents = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"  Post: {post}")
    print(f"  WORK_OF_ART: {works if works else 'NONE'}")
    print(f"  All entities: {all_ents}")
    print()

SpaCy WORK_OF_ART extraction:

  Post: I watched Severence yesterday and it was mid
  WORK_OF_ART: ['Severence']
  All entities: [('Severence', 'WORK_OF_ART'), ('yesterday', 'DATE')]

  Post: Cant wait to watch spider man tomorrow
  WORK_OF_ART: NONE
  All entities: [('spider man', 'PERSON'), ('tomorrow', 'DATE')]

  Post: We finally watched Barbie lastg night and it was great!!!
  WORK_OF_ART: ['Barbie']
  All entities: [('Barbie', 'WORK_OF_ART'), ('lastg night', 'TIME')]

  Post: Anyone have movie recommendations for tonight?
  WORK_OF_ART: NONE
  All entities: [('tonight', 'TIME')]

  Post: That film was incredible, no notes, 10/10
  WORK_OF_ART: NONE
  All entities: [('10/10', 'CARDINAL')]

  Post: Like my photo!? Shot on 35mm film
  WORK_OF_ART: NONE
  All entities: [('35mm', 'QUANTITY')]

  Post: Make your favourite film sound as boring as possible
  WORK_OF_ART: NONE
  All entities: []

  Post: Finally watched sopranos
  WORK_OF_ART: ['sopranos']
  All entities: [('sopranos', 'W

In [ ]:
# Run NER on all gold-labelled positive posts
df = pd.read_csv("../data/backfill_watched_gold_labelled.csv")
positives = df[df["gold_label"] == 1]
print(f"running SpaCy NER on {len(positives)} positive posts...\n")

ner_results = []
for idx, row in positives.iterrows():
    doc = nlp(row["text"])
    works = [ent.text for ent in doc.ents if ent.label_ == "WORK_OF_ART"]
    all_ents = [(ent.text, ent.label_) for ent in doc.ents]
    ner_results.append({
        "text": row["text"],
        "work_of_art": works,
        "all_entities": all_ents,
        "has_WORK_OF_ART": len(works) > 0,
    })

ner_df = pd.DataFrame(ner_results)

# analysis
num_found = ner_df["has_WORK_OF_ART"].sum()
num_missed = len(ner_df) - num_found
print(f"WORK_OF_ART found: {num_found} / {len(ner_df)} ({100*num_found/len(ner_df):.0f}%)")
print(f"No WORK_OF_ART found: {num_missed} / {len(ner_df)} ({100*num_missed/len(ner_df):.0f}%)")

# print posts where NER found a WORK_OF_ART
print("POSTS WITH EXTRACTED WORK_OF_ART:")
for idx, row in ner_df[ner_df["has_WORK_OF_ART"]].iterrows():
    print(f"\n  WORK_OF_ART: {row['work_of_art']}")
    print(f"  Post:  {row['text']}")

# Show posts where NER found nothing
print("POSTS WITH NO WORK_OF_ART EXTRACTED:")
#using ~ to invert the condition (like NOT)
for idx, row in ner_df[~ner_df["has_WORK_OF_ART"]].iterrows():
    print(f"\n  All ents: {row['all_entities']}")
    print(f"  Post:     {row['text']}")

running SpaCy NER on 209 positive posts...

WORK_OF_ART found: 97 / 209 (46%)
No WORK_OF_ART found: 112 / 209 (54%)
POSTS WITH EXTRACTED WORK_OF_ART:

  WORK_OF_ART: ['Caroushell', 'Frogman']
  Post:  Yesterday I watched a movie called Caroushell (brought to my attention by my friend who also watched it) and it was an Experience. Today I am watching a movie called Frogman.

"Are you saying frogman fucks?" Incredible dialogue.

  WORK_OF_ART: ['Insidious']
  Post:  The latest Insidious movie gave me everything I wanted and more! Creepy and fun and full of emotion. Family and love and embracing power and fighting the dead and Elise!!!!

  WORK_OF_ART: ['Thor']
  Post:  The thing is, many of them have themes! Often perfectly fine themes! None of which are ever used outside their individual movies and some of which change with each movie - Thor has a different individual theme in each of his solo films!

  WORK_OF_ART: ['Spiderverse']
  Post:  This is one of the many things the animated Sp

In [12]:
#test NER WORK_OF_ART results against the movie database (using a personal API TMDB_READ_ACCESS_TOKEN)
# API docs: https://developer.themoviedb.org/reference/search-multi
TMDB_SEARCH_URL = "https://api.themoviedb.org/3/search/multi"

#function to query the movie database API using a WORK_OF_ART string
#using httpx instead of requests library with an eye on using async via httpx down the line
def search_tmdb(WORK_OF_ART):
    #search the movie database using WORK_OF_ART, returns the most popular matching movie/tv show or nothing
    headers = {
        "Authorization": f"Bearer {TMDB_API_KEY}",
        "accept": "application/json",
    }
    #send GET request to API, with a 10s timeout before raising error
    resp = httpx.get(TMDB_SEARCH_URL, headers=headers, params={"query": WORK_OF_ART}, timeout=10)
    #raise error code
    resp.raise_for_status()

    #filter the API response on movie and tv show - the search/multi endpoint also includes people, but this isnt relevant 
    results = [r for r in resp.json().get("results", []) if r.get("media_type") in ("movie", "tv")]
    if not results:
        return None

    #select the top result based on popularity (this is defined by themoviedb here:https://starlight.mintlify.app/api-reference/popularandtrending)
    top = results[0]

    #return result as dictionary
    return {
        "tmdb_id": top.get("id"),
        #get title or name, depending on movie or tv show
        "tmdb_title": top.get("title") or top.get("name", ""),
        "media_type": top.get("media_type"),
        #get release date or first air date depending on movie or tv show
        "release_date": top.get("release_date") or top.get("first_air_date", ""),
        #get average vote data for some extra context, might use it later
        "vote_average": top.get("vote_average", 0),
    }

In [ ]:
# test on set of records
sample = ner_df[ner_df["has_WORK_OF_ART"]].head(100)

print(f"Testtestinging moviedb api linking on {len(sample)} posts:\n")
for idx, row in sample.iterrows():
    print(f"Post: {row['text']}")
    for title in row["work_of_art"]:
        result = search_tmdb(title)
        #sleep for courtesy delay. Docs says rate limit of 40 per second so this is well within range
        time.sleep(0.20)
        if result:
            year = result["release_date"][:4] if result["release_date"] else "N/A"
            print(f"  '{title}' = {result['tmdb_title']} ({result['media_type']}, {year}, rating: {result['vote_average']})")
        else:
            print(f"  '{title}' = NO MATCH")
    print()

Testtestinging moviedb api linking on 97 posts:

Post: Yesterday I watched a movie called Caroushell (brought to my attention by my friend who also watched it) and it was an Experience. Today I am watching a movie called Frogman.

"Are you saying frogman fucks?" Incredible dialogue.
  'Caroushell' -> CarousHELL (movie, 2016, rating: 4.364)
  'Frogman' -> Frogman Returns (movie, 2025, rating: 0.0)

Post: The latest Insidious movie gave me everything I wanted and more! Creepy and fun and full of emotion. Family and love and embracing power and fighting the dead and Elise!!!!
  'Insidious' -> Insidious (movie, 2011, rating: 6.953)

Post: The thing is, many of them have themes! Often perfectly fine themes! None of which are ever used outside their individual movies and some of which change with each movie - Thor has a different individual theme in each of his solo films!
  'Thor' -> Thor (movie, 2011, rating: 6.771)

Post: This is one of the many things the animated Spiderverse movies nail